# AI 錄音逐字稿處理 Pipeline v2

## 流程說明
```
Step 1  │ Whisper 轉錄              → {BASE_NAME}_1_原始逐字稿.txt
Step 2  │ Gemini 音節修復 (保留冗詞) → {BASE_NAME}_2_音節修復保留冗詞版.md
Step 3  │ ⚠️ 人工聽音修復 (手動)
Step 4  │ Gemini 刪除冗詞           → {BASE_NAME}_4_刪除冗詞版.md
Step 5  │ ⚠️ 人工確認 (手動)
Step 6  │ Gemini 時間序局部摘要      → {BASE_NAME}_6_時間序局部摘要.md
Step 7  │ ⚠️ 人工補充摘要 (手動)
Step 8  │ Gemini 最終重構摘要        → {BASE_NAME}_8_最終重構摘要.md
```

## 使用方式
1. 修改下方 Cell 0 的 `BASE_NAME` 與 `MEETING_CONTEXT`
2. 確保音檔已放到 Google Drive 的 `AI_transcribe` 資料夾
3. 依序執行各 Cell，遇到人工步驟時請先處理再繼續

In [ ]:
# ============================================================
# Cell 0：【每次使用前修改這裡】專案設定
# ============================================================

# 檔案基本名稱（音檔需命名為 {BASE_NAME}.m4a 或 .mp3）
BASE_NAME = "NLP_Group5_Meeting"

# 這場會議的背景脈絡（讓 Gemini 修復時更準確）
MEETING_CONTEXT = """
這是一場資工所「NLP 期末專案」的進度報告與助教討論。
專案主題：情緒語氣是否會干擾 LLM 的事實查核（Fact-Checking）邏輯判斷。
專有名詞清單：Ground Truth、Baseline、Few-shot Prompting、LLM、Prompt、
CoFED（Cofacts）資料集、Fake、True、Neutral、Flip（翻轉）、
Atomic Facts、Semantic Similarity、Binomial Test（二項式檢定）、
p-value、Rule-based、Hallucination（幻覺）、NVIDIA、
Public Health、Scam、Category、Confidence Score。
語者：主要有學生（報告者）與助教（提問者）。
"""

# Gemini 模型選擇（優先使用 gemini-2.5-flash，穩定且便宜）
# 可換成 "gemini-2.5-pro" 若需要更高品質但費用較高
LLM_MODEL_NAME = "gemini-2.5-flash"

# Google Drive 路徑設定
DRIVE_DIR = f"/content/drive/MyDrive/AI_transcribe"

print(f"✅ 設定完成")
print(f"   BASE_NAME    : {BASE_NAME}")
print(f"   LLM Model    : {LLM_MODEL_NAME}")
print(f"   Drive Dir    : {DRIVE_DIR}")

In [ ]:
# ============================================================
# Cell 1：環境安裝與初始化（每次 runtime 重啟後需執行）
# ============================================================

!pip install -q faster-whisper google-generativeai

import os
import time
import textwrap
from google.colab import drive, userdata
import google.generativeai as genai

# 掛載 Drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

# 定義所有檔案路徑
AUDIO_EXTENSIONS = [".m4a", ".mp3", ".wav", ".ogg"]
AUDIO_FILE_PATH = None
for ext in AUDIO_EXTENSIONS:
    candidate = os.path.join(DRIVE_DIR, f"{BASE_NAME}{ext}")
    if os.path.exists(candidate):
        AUDIO_FILE_PATH = candidate
        break

PATH_1_RAW      = os.path.join(DRIVE_DIR, f"{BASE_NAME}_1_原始逐字稿.txt")
PATH_2_REPAIR   = os.path.join(DRIVE_DIR, f"{BASE_NAME}_2_音節修復保留冗詞版.md")
PATH_4_CLEAN    = os.path.join(DRIVE_DIR, f"{BASE_NAME}_4_刪除冗詞版.md")
PATH_6_CHRONO   = os.path.join(DRIVE_DIR, f"{BASE_NAME}_6_時間序局部摘要.md")
PATH_8_FINAL    = os.path.join(DRIVE_DIR, f"{BASE_NAME}_8_最終重構摘要.md")

# 設定 Gemini API
# 請在 Colab 左側 🔑 Secrets 新增 GEMINI_API_KEY
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

print(f"✅ 環境初始化完成")
if AUDIO_FILE_PATH:
    print(f"   🎵 找到音檔: {AUDIO_FILE_PATH}")
else:
    print(f"   ⚠️  找不到音檔，請確認 {DRIVE_DIR} 內有 {BASE_NAME}.m4a（或 .mp3）")

# 輔助函式：呼叫 Gemini（含 retry 與錯誤處理）
def call_gemini(system_prompt, user_text, temperature=0.1, max_retries=3):
    """呼叫 Gemini API，自動重試，回傳回應文字。"""
    model = genai.GenerativeModel(
        model_name=LLM_MODEL_NAME,
        system_instruction=system_prompt
    )
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                user_text,
                generation_config=genai.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=65536
                )
            )
            return response.text
        except Exception as e:
            print(f"   ⚠️  嘗試 {attempt+1}/{max_retries} 失敗: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
    raise RuntimeError("Gemini API 多次失敗，請檢查 API Key 與模型名稱。")

# 輔助函式：長文字分塊處理（避免超過 context window）
def chunk_text(text, max_chars=60000):
    """將長文字依時間標記分塊，避免切斷同一段落。"""
    if len(text) <= max_chars:
        return [text]
    lines = text.split('\n')
    chunks, current_chunk, current_len = [], [], 0
    for line in lines:
        if current_len + len(line) > max_chars and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk, current_len = [], 0
        current_chunk.append(line)
        current_len += len(line) + 1
    if current_chunk:
        chunks.append('\n'.join(current_chunk))
    print(f"   📦 文字太長，已分成 {len(chunks)} 塊處理")
    return chunks

def process_with_chunking(system_prompt, text, temperature=0.1):
    """分塊呼叫 Gemini 並合併結果。"""
    chunks = chunk_text(text)
    results = []
    for i, chunk in enumerate(chunks):
        print(f"   🔄 處理第 {i+1}/{len(chunks)} 塊...")
        prompt = f"（這是第 {i+1}/{len(chunks)} 段，請依指示處理）\n\n{chunk}"
        results.append(call_gemini(system_prompt, prompt, temperature))
        if len(chunks) > 1:
            time.sleep(2)  # 避免 rate limit
    return '\n\n---（分塊接續）---\n\n'.join(results) if len(chunks) > 1 else results[0]

In [ ]:
# ============================================================
# Cell 2：[Step 1] Whisper 強效轉錄
# ============================================================
# 輸出：{BASE_NAME}_1_原始逐字稿.txt

if os.path.exists(PATH_1_RAW):
    char_count = len(open(PATH_1_RAW, encoding='utf-8').read())
    print(f"✅ 已有 [1_原始逐字稿]，跳過轉錄（{char_count:,} 字）")
    print(f"   路徑: {PATH_1_RAW}")
    print(f"   若要重新轉錄，請手動刪除此檔案後再執行。")
else:
    if not AUDIO_FILE_PATH:
        raise FileNotFoundError(f"找不到音檔！請確認 {DRIVE_DIR} 內有 {BASE_NAME}.m4a")

    print("🚀 開始 Whisper large-v3 轉錄（需要 A100 GPU，約需 1-3 分鐘）...")
    from faster_whisper import WhisperModel

    model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    initial_prompt = f"""
    {MEETING_CONTEXT}
    請盡量辨識每個字，包括口語贅字「呃」「嗯」「然後」「對」「就是說」。
    """.strip()

    # word_timestamps=True 是關鍵：強制 Whisper 逐字處理，不會整段跳過
    # vad_filter=False：完全關掉語音偵測，讓所有音頻都被轉錄
    segments, info = model.transcribe(
        AUDIO_FILE_PATH,
        beam_size=5,
        language="zh",
        vad_filter=False,
        word_timestamps=True
    )

    raw_text = ""
    seg_count = 0
    for segment in segments:
        # 用 word_timestamps 把文字串回來，確保每個字都被處理到
        if segment.words:
            words_text = "".join([w.word for w in segment.words]).strip()
        else:
            words_text = segment.text.strip()
        line = f"[{segment.start:.1f}s - {segment.end:.1f}s] {words_text}\n"
        raw_text += line
        seg_count += 1
        if seg_count <= 5 or seg_count % 50 == 0:
            print(line.strip())

    with open(PATH_1_RAW, "w", encoding="utf-8") as f:
        f.write(raw_text)

    print(f"\n💾 [Step 1 完成] 原始逐字稿已儲存")
    print(f"   共 {seg_count} 段，{len(raw_text):,} 字")
    print(f"   路徑: {PATH_1_RAW}")
    print(f"   最後時間戳：{raw_text.strip().split(chr(10))[-1][:20]}...")


In [ ]:
# ============================================================
# Cell 3：[Step 2] 沉浸式音節修復（嚴格保留所有冗詞）
# ============================================================
# 輸出：{BASE_NAME}_2_音節修復保留冗詞版.md
# 之後需要人工聽音修復（Step 3）

if os.path.exists(PATH_2_REPAIR):
    char_count = len(open(PATH_2_REPAIR, encoding='utf-8').read())
    print(f"✅ 已有 [2_音節修復版]，跳過（{char_count:,} 字）")
    print(f"   ⚠️  請確認你已完成 Step 3（人工聽音修復）再繼續！")
else:
    print("🧠 開始音節修復（嚴格保留冗詞版）...")

    with open(PATH_1_RAW, "r", encoding="utf-8") as f:
        raw_text = f.read()

    original_char_count = len(raw_text)
    print(f"   原始逐字稿：{original_char_count:,} 字")

    system_prompt = f"""
你是一位極度嚴謹的語音辨識修復員。

【背景知識】
{MEETING_CONTEXT}

【絕對禁止事項 - 違反即為失敗】
❌ 禁止刪除任何字詞，包括「呃」「嗯」「然後」「那個」「就是說」「對」等發語詞
❌ 禁止刪除講者的結巴、重複、自我修正
❌ 禁止摘要、精簡、合併相似語句
❌ 禁止增加原文沒有的任何內容

【唯一允許的操作：音節修復】
✅ 修正 ASR 聽錯的同音異義詞（例如：「光速」→「Ground Truth」、「恩囉比」→「NLP」、「scan」→「Scam」）
✅ 修正明顯的專有名詞拼音問題
✅ 辨識語者（[講者A]=[助教]，[講者B]=[學生報告者]），並合併同一語者連續發言的時間軸
✅ 保留並整理時間標記格式：[XXXs - XXXs] [講者X]：...

【品質驗證標準】
修復後的總字數應接近原始逐字稿（允許 ±10% 的浮動，主要來自修正專有名詞字數變化）。
若你發現自己刪掉了超過 10% 的內容，代表你違反了規則。

【格式範例】
[0.0s - 15.5s] [講者B]：呃... 我們會先講一下我們的實驗流程，我們是用那個 CoFED 這個資料集，然後...
[15.5s - 22.0s] [講者A]：那你們跟情緒的關係是怎樣？
""".strip()

    result = process_with_chunking(system_prompt, raw_text, temperature=0.05)

    repaired_char_count = len(result)
    retention_rate = repaired_char_count / original_char_count * 100

    with open(PATH_2_REPAIR, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"\n✨ [Step 2 完成] 音節修復版已儲存")
    print(f"   原始：{original_char_count:,} 字 → 修復後：{repaired_char_count:,} 字")
    print(f"   保留率：{retention_rate:.1f}%", end="")
    if retention_rate < 85:
        print(f" ⚠️  保留率偏低！Gemini 可能刪掉了太多內容，建議重新執行或人工補充")
    elif retention_rate < 95:
        print(f" ✅  （正常範圍）")
    else:
        print(f" 🎉 （保留完整）")
    print(f"\n   路徑: {PATH_2_REPAIR}")
    print(f"\n📋 [Step 3] 請打開上面的檔案，對照錄音進行人工聽音修復，確認無誤後再執行 Cell 4。")

In [ ]:
# ============================================================
# Cell 4：[Step 4] 刪除冗詞（讀取人工修復後的檔案）
# ============================================================
# 輸入：{BASE_NAME}_2_音節修復保留冗詞版.md（你已人工修復的版本）
# 輸出：{BASE_NAME}_4_刪除冗詞版.md

if os.path.exists(PATH_4_CLEAN):
    char_count = len(open(PATH_4_CLEAN, encoding='utf-8').read())
    print(f"✅ 已有 [4_刪除冗詞版]，跳過（{char_count:,} 字）")
else:
    if not os.path.exists(PATH_2_REPAIR):
        raise FileNotFoundError("請先完成 Step 2 與 Step 3（人工修復）！")

    print("🧠 開始清洗口語冗詞（保留所有實質內容）...")

    with open(PATH_2_REPAIR, "r", encoding="utf-8") as f:
        repaired_text = f.read()

    system_prompt = f"""
你是一位專業的學術會議記錄編輯。

【背景知識】
{MEETING_CONTEXT}

【任務：清洗冗詞，但絕對不可摘要】
✅ 可以刪除：純粹的發語詞（「呃」「嗯」單獨出現時）、講者明顯的結巴重複（「然後然後然後」→「然後」）
✅ 可以修正：讓句子更通順的語法調整
❌ 禁止摘要或合併：任何助教的提問、學生的回答、技術細節、數據、建議，都必須完整保留
❌ 禁止刪除：有實質意義的「然後」「對」（作為確認語）、轉折詞、問答的完整來回

【格式要求】
保留時間標記與語者標記。格式：[XXXs - XXXs] [講者X]：...
""".strip()

    result = process_with_chunking(system_prompt, repaired_text, temperature=0.1)

    with open(PATH_4_CLEAN, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"\n✨ [Step 4 完成] 刪除冗詞版已儲存")
    print(f"   路徑: {PATH_4_CLEAN}")
    print(f"\n📋 [Step 5] 請快速確認此版本的內容完整性，再執行 Cell 5。")

In [ ]:
# ============================================================
# Cell 5：[Step 6] 依時間序的局部摘要
# ============================================================
# 輸出：{BASE_NAME}_6_時間序局部摘要.md
# 之後你可以對照此版本進行人工補充（Step 7）

if os.path.exists(PATH_6_CHRONO):
    print(f"✅ 已有 [6_時間序局部摘要]，跳過")
    print(f"   ⚠️  請確認你已完成 Step 7（人工補充摘要）再繼續！")
else:
    if not os.path.exists(PATH_4_CLEAN):
        raise FileNotFoundError("請先完成 Step 4！")

    print("🧠 依時間軸生成局部摘要...")

    with open(PATH_4_CLEAN, "r", encoding="utf-8") as f:
        clean_text = f.read()

    system_prompt = f"""
你是一位精準的學術會議記錄助理。

【背景知識】
{MEETING_CONTEXT}

【任務：依時間序製作局部摘要】
1. 順著時間軸，每當話題轉換時，切分一個段落
2. 每個段落標註時間區間，例如 [0.0s - 120.0s]
3. 以條列式寫出該區間的：主要討論內容、助教提問或質疑、具體建議
4. 關鍵數字（題數、p-value、模型大小等）與技術細節一律保留，不可模糊化
5. 每個段落都要有「誰說了什麼」的脈絡

【注意】
這份摘要是給人工對照確認用的，所以寧可詳細也不要精簡。
""".strip()

    result = process_with_chunking(system_prompt, clean_text, temperature=0.2)

    with open(PATH_6_CHRONO, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"\n✨ [Step 6 完成] 時間序局部摘要已儲存")
    print(f"   路徑: {PATH_6_CHRONO}")
    print(f"\n📋 [Step 7] 請打開此檔案，對照你自己的記憶或逐字稿補充遺漏的重點，再執行 Cell 6。")

In [ ]:
# ============================================================
# Cell 6：[Step 8] 解析重構：最終綜合會議紀錄
# ============================================================
# 輸入：{BASE_NAME}_6_時間序局部摘要.md（你人工確認過的版本）
# 輸出：{BASE_NAME}_8_最終重構摘要.md

if os.path.exists(PATH_8_FINAL):
    print(f"✅ 已有 [8_最終重構摘要]，全部流程完成！")
    print(f"   路徑: {PATH_8_FINAL}")
else:
    if not os.path.exists(PATH_6_CHRONO):
        raise FileNotFoundError("請先完成 Step 6 與 Step 7！")

    print("🧠 重構最終綜合會議紀錄...")

    with open(PATH_6_CHRONO, "r", encoding="utf-8") as f:
        chrono_text = f.read()

    system_prompt = f"""
你是一位資深技術專案經理，擅長撰寫學術研究的結構化會議紀錄。

【背景知識】
{MEETING_CONTEXT}

【任務：打破時間軸，重構為結構化最終報告】
請根據提供的時間序摘要，整合成一份專業的最終會議紀錄。
相同主題的討論（即使在不同時間段）應整合在同一章節下。

【必要的 Markdown 章節結構】
# [會議主題] 會議紀錄
**日期**：（如已知）
**參與者**：學生（報告者）、助教

## 1. 實驗流程與資料集改寫
## 2. 資料品質評估方法
## 3. 實驗數據與統計分析結果
## 4. 助教回饋與建議
## 5. 下一步行動 (Action Items)

【語氣要求】
學術嚴謹，保留所有技術細節與數字，以繁體中文撰寫。
""".strip()

    # 最終摘要通常不需要分塊，但以防萬一
    result = call_gemini(system_prompt, chrono_text, temperature=0.2)

    with open(PATH_8_FINAL, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"\n🎉 [Step 8 完成] 最終重構報告已儲存！")
    print(f"   路徑: {PATH_8_FINAL}")

In [ ]:
# ============================================================
# Cell 7：[工具] 強制重新執行某個 Step
# ============================================================
# 若某個 Step 的結果不滿意，修改下面的 STEP_TO_REDO 再執行此 Cell
# 它會刪除對應檔案，讓你重新跑那個 Step

STEP_TO_REDO = None  # 填入 1, 2, 4, 6 或 8；None 表示不做任何事

step_to_path = {
    1: PATH_1_RAW,
    2: PATH_2_REPAIR,
    4: PATH_4_CLEAN,
    6: PATH_6_CHRONO,
    8: PATH_8_FINAL,
}

if STEP_TO_REDO is None:
    print("ℹ️  STEP_TO_REDO = None，無任何動作。")
elif STEP_TO_REDO in step_to_path:
    target = step_to_path[STEP_TO_REDO]
    if os.path.exists(target):
        os.remove(target)
        print(f"🗑️  已刪除 Step {STEP_TO_REDO} 的輸出檔案：{target}")
        print(f"   請重新執行對應的 Cell 以重跑此 Step。")
    else:
        print(f"ℹ️  Step {STEP_TO_REDO} 的檔案不存在，不需要刪除。")
else:
    print(f"⚠️  無效的 STEP_TO_REDO 值：{STEP_TO_REDO}。請填入 1, 2, 4, 6 或 8。")

In [ ]:
# ============================================================
# Cell 8：[工具] 查看所有檔案狀態
# ============================================================

print("📊 Pipeline 狀態總覽")
print("=" * 60)

steps = [
    (1, "原始逐字稿 (Whisper)",            PATH_1_RAW),
    (2, "音節修復保留冗詞版 (Gemini)",       PATH_2_REPAIR),
    ("3", "⚠️  人工聽音修復",               None),
    (4, "刪除冗詞版 (Gemini)",              PATH_4_CLEAN),
    ("5", "⚠️  人工確認",                   None),
    (6, "時間序局部摘要 (Gemini)",           PATH_6_CHRONO),
    ("7", "⚠️  人工補充摘要",               None),
    (8, "最終重構摘要 (Gemini)",             PATH_8_FINAL),
]

for step_num, step_name, path in steps:
    if path is None:
        print(f"   Step {step_num} │ {step_name}")
    elif os.path.exists(path):
        size = os.path.getsize(path)
        char_count = len(open(path, encoding='utf-8').read())
        print(f"   Step {step_num} │ ✅ {step_name}")
        print(f"          │    {char_count:,} 字 | {size/1024:.1f} KB")
    else:
        print(f"   Step {step_num} │ ⏳ {step_name}（尚未產出）")